In [1]:
import os
import re
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import hvplot.pandas
import holoviews as hv
import glob

## Downloading precipitation station data from MeteoSwiss

In [137]:
import os
import requests

# The starting API endpoint
START_URL = "https://data.geo.admin.ch/api/stac/v1/collections/ch.meteoschweiz.ogd-smn-precip/items"
DOWNLOAD_DIR = "meteoswiss_data"

current_url = START_URL
page_count = 1

while current_url:
    print(f"\n--- Processing Page {page_count} ---")
    print(f"Fetching: {current_url}")
    
    response = requests.get(current_url)
    response.raise_for_status()
    data = response.json()

    features = data.get("features", [])
    print(f"Found {len(features)} stations on this page.")

    for feature in features:
        assets = feature.get("assets", {})
        for asset_key, asset_value in assets.items():
            href = asset_value.get("href")
            # Ensure it's a CSV file
            if href and href.lower().endswith(".csv") and "_h_historical" in href:
                file_name = os.path.basename(href)
                file_path = os.path.join(DOWNLOAD_DIR, file_name)

                # Simple check to avoid re-downloading if script restarts
                if os.path.exists(file_path):
                    continue

                try:
                    df = pd.read_csv(href, sep=';')
                    df.to_csv(f'~/hyclimm/Rhone/meteo_stations/{file_name}')
                except Exception as e:
                    print(f"Error downloading {file_name}: {e}")

    # PAGINATION LOGIC: Look for the 'next' link
    links = data.get("links", [])
    next_link = next((link for link in links if link.get("rel") == "next"), None)
    
    if next_link:
        current_url = next_link.get("href")
        page_count += 1
    else:
        current_url = None
        print("\nNo more pages found. Download complete.")


--- Processing Page 1 ---
Fetching: https://data.geo.admin.ch/api/stac/v1/collections/ch.meteoschweiz.ogd-smn-precip/items
Found 100 stations on this page.

--- Processing Page 2 ---
Fetching: https://data.geo.admin.ch/api/stac/v1/collections/ch.meteoschweiz.ogd-smn-precip/items?cursor=cD12c2Jhcw%3D%3D
Found 41 stations on this page.

No more pages found. Download complete.


## Data Formatting

### Rhone domain

In [ ]:
def read_station_data(dat_file):
    df0 = pd.read_csv(dat_file, sep="\t", header=4, na_values=-9999.)
    data = df0.iloc[:, 4:].apply(pd.to_numeric, errors='coerce')
    time = df0.iloc[:, :4].astype(int)
    dt = pd.DataFrame({
        'year': time['YY'],
        'month': time['MM'],
        'day': time['DD']
    })
    data.loc[:, 'datetime'] = pd.to_datetime(dt)
    data['datetime'] = data['datetime'] + pd.to_timedelta(time['HH'].astype(int)-1, unit='h')
    data.set_index('datetime', inplace=True)
    data.index.name = 'time'

    with open(dat_file, 'r') as f:
        header = [next(f) for _ in range(4)]
    return data, header

In [ ]:
dat_file = '/home/aelkouk/hyclimm/Rhone/Input/timeseries/precipitation_reg2_cleaned.dat'
with open(dat_file, 'r') as f:
    f.readline() # skip title
    header = [next(f).split()[4:] for _ in range(4)]
rhone_stations_reg2 = pd.DataFrame(np.array(header[:-1]).T, index=header[-1], columns=['z','x','y'])
df_reg2, header_reg2 = read_station_data(dat_file)

dat_file = '/home/aelkouk/hyclimm/Rhone/Input/timeseries/precipitation_reg1_cleaned.dat'
with open(dat_file, 'r') as f:
    f.readline() # skip title
    header = [next(f).split()[4:] for _ in range(4)]
rhone_stations_reg1 = pd.DataFrame(np.array(header[:-1]).T, index=header[-1], columns=['z','x','y'])
df_reg1, header_reg1 = read_station_data(dat_file)

rhone_stations = pd.concat([rhone_stations_reg1, rhone_stations_reg2])

In [226]:
gdf_rhone_stations = gpd.GeoDataFrame(rhone_stations, geometry=gpd.points_from_xy(rhone_stations.x, rhone_stations.y), crs="EPSG:21781").to_crs("EPSG:4326")
gdf_rhone_stations['lon'] = gdf_rhone_stations.geometry.x
gdf_rhone_stations['lat'] = gdf_rhone_stations.geometry.y

In [212]:
# Rhone extent
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:21781", "EPSG:4326", always_xy=True)

x1, y1, x2, y2 = 548000, 78000, 680500, 168500

lon1, lat1 = transformer.transform(x1, y1)
lon2, lat2 = transformer.transform(x2, y2)

buffer = 0.05
lon1, lat1, lon2, lat2 = lon1 - buffer, lat1 - buffer, \
                         lon2 + buffer, lat2 + buffer, 
print(f"Lon bounds: {lon1:.4f}, {lon2:.4f}")
print(f"Lat bounds: {lat1:.4f}, {lat2:.4f}")

Lon bounds: 6.7193, 8.5405
Lat bounds: 45.8016, 46.7129


In [213]:
stations_meteoch = pd.read_csv('/home/aelkouk/hyclimm/Rhone/meteo_stations/meteoswiss/ogd-smn-precip_meta_stations.csv', sep=';', encoding='latin1', index_col='station_abbr')
x, y = stations_meteoch['station_coordinates_wgs84_lon'], stations_meteoch['station_coordinates_wgs84_lat']
mask = (x>=lon1)&(x<=lon2)&(y>=lat1)&(y<=lat2)
print(mask.sum())

52


In [228]:
import geoviews as gv
interactive_map1 = stations_meteoch[mask].hvplot.points(
    x='station_coordinates_wgs84_lon', 
    y='station_coordinates_wgs84_lat', 
    geo=True, 
    tiles='EsriTerrain',
    features=['rivers', 'lakes'], # This overlays water features
    color='red',
    size=50,
    title='Stations with Hydrologic Features'
)
interactive_map2 = gdf_rhone_stations.hvplot.points(
    x='lon', 
    y='lat', 
    geo=True, 
    tiles='EsriTerrain',
    features=['rivers', 'lakes'], # This overlays water features
    color='blue',
    size=10,
    title='Stations with Hydrologic Features'
)
interactive_map = interactive_map1 * interactive_map2
interactive_map

:Overlay
   .WMTS.I    :WMTS   [Longitude,Latitude]
   .Points.I  :Points   [station_coordinates_wgs84_lon,station_coordinates_wgs84_lat]
   .Lakes.I   :Feature   [Longitude,Latitude]
   .Rivers.I  :Feature   [Longitude,Latitude]
   .WMTS.II   :WMTS   [Longitude,Latitude]
   .Points.II :Points   [lon,lat]
   .Lakes.II  :Feature   [Longitude,Latitude]
   .Rivers.II :Feature   [Longitude,Latitude]

### Formatting

In [783]:
# Rhone extent
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:21781", "EPSG:4326", always_xy=True)

x1, y1, x2, y2 = 548000, 78000, 680500, 168500

lon1, lat1 = transformer.transform(x1, y1)
lon2, lat2 = transformer.transform(x2, y2)

buffer = 0.05
lon1, lat1, lon2, lat2 = lon1 - buffer, lat1 - buffer, \
                         lon2 + buffer, lat2 + buffer, 
print(f"Lon bounds: {lon1:.4f}, {lon2:.4f}")
print(f"Lat bounds: {lat1:.4f}, {lat2:.4f}")

Lon bounds: 6.7193, 8.5405
Lat bounds: 45.8016, 46.7129


In [ ]:
varname = 'rre150h0'

In [954]:
stations_meteoch = pd.read_csv('/home/aelkouk/hyclimm/Rhone/meteo_stations/meteoswiss/ogd-smn-precip_meta_stations.csv', sep=';', encoding='latin1', index_col='station_abbr')
x, y = stations_meteoch['station_coordinates_wgs84_lon'], stations_meteoch['station_coordinates_wgs84_lat']
mask = (x>=lon1)&(x<=lon2)&(y>=lat1)&(y<=lat2)
print(mask.sum())

stations_meteoch_rhone = stations_meteoch[mask]

52


In [955]:
datetime = pd.date_range(start='1991-01-01 00:00:00', 
                           end='2025-12-31 23:00:00', 
                           freq='h')
df_prec = pd.DataFrame(index=datetime)

In [956]:
for station in stations_meteoch_rhone.index:
    datafiles = glob.glob(f'/home/aelkouk/hyclimm/Rhone/meteo_stations/meteoswiss/ogd-smn-precip_{station.lower()}_h_*')
    if len(datafiles)>1:
        df0 = pd.concat([pd.read_csv(datafile, index_col=0) for datafile in datafiles])
    else:
        df0 = pd.read_csv(datafiles[0], index_col=0)
    df0['datetime'] = pd.to_datetime(df0['reference_timestamp'], format='%d.%m.%Y %H:%M')
    df0 = df0.set_index('datetime')
    df_prec.loc[:, station] = df0[varname]
    print(station, ':')
    print('\t', df0.index[0], df0.index[-1], (df_prec[station]>0).sum())

AVA :
	 2016-03-17 10:00:00 2025-12-31 23:00:00 11694
BEX :
	 2012-12-19 16:00:00 2025-12-31 23:00:00 13379
BOS :
	 2014-09-16 08:00:00 2025-12-31 23:00:00 14460
BRI :
	 2013-10-16 10:00:00 2025-12-31 23:00:00 8623
FIO :
	 2013-09-04 08:00:00 2025-12-31 23:00:00 14945
FIT :
	 2016-02-09 10:00:00 2025-12-31 23:00:00 10268
GOA :
	 2013-10-16 10:00:00 2025-12-31 23:00:00 20305
GSG :
	 2014-09-24 08:00:00 2025-12-31 23:00:00 16084
GTT :
	 2014-01-16 06:00:00 2025-12-31 23:00:00 17408
KAS :
	 2015-11-04 10:00:00 2025-12-31 23:00:00 13596
KIE :
	 2016-01-01 00:00:00 2025-12-31 23:00:00 12755
LEU :
	 2013-09-04 08:00:00 2025-12-31 23:00:00 13812
LTB :
	 2015-09-16 08:00:00 2025-12-31 23:00:00 12024
MAB :
	 2016-03-30 08:00:00 2025-12-31 23:00:00 8921
ORS :
	 2012-12-21 15:00:00 2025-12-31 23:00:00 12112
ROM :
	 2016-03-18 10:00:00 2025-12-31 23:00:00 9485
VSANZ :
	 2010-10-06 00:00:00 2025-12-31 23:00:00 17708
VSARO :
	 2011-10-05 15:00:00 2025-12-31 23:00:00 17464
VSBAS :
	 2010-10-06 09:00:

In [958]:
dat_file = '/home/aelkouk/hyclimm/Rhone/Input/timeseries/precipitation_reg2_cleaned.dat'
with open(dat_file, 'r') as f:
    f.readline() # skip title
    header = [next(f).split()[4:] for _ in range(4)]
rhone_stations_reg2 = pd.DataFrame(np.array(header[:-1]).T, index=header[-1], columns=['z','x','y']).apply(pd.to_numeric)
df_reg2, header_reg2 = read_station_data(dat_file)

In [959]:
dat_file = '/home/aelkouk/hyclimm/Rhone/Input/timeseries/precipitation_reg1_cleaned.dat'
with open(dat_file, 'r') as f:
    f.readline() # skip title
    header = [next(f).split()[4:] for _ in range(4)]
rhone_stations_reg1 = pd.DataFrame(np.array(header[:-1]).T, index=header[-1], columns=['z','x','y']).apply(pd.to_numeric)
df_reg1, header_reg1 = read_station_data(dat_file)

In [960]:
duplicates = [st for st in rhone_stations_reg2.index if st.endswith('.1')]
print(len([st for st in duplicates if st[:-2] in rhone_stations_reg2.index]), len(duplicates))
unique = [st for st in rhone_stations_reg2.index if not st.endswith('.1') and not st.startswith('SLF')]
print(len(unique))

df_reg2_updated = pd.DataFrame(index=df_reg2.index)
for st1 in duplicates:
    st0 = st1[:-2]
    data = df_reg2[st0]
    data[data.isnull()] = df_reg2[st1]
    df_reg2_updated.loc[:, st0] = data
df_reg2_updated.loc[:, unique] = df_reg2.loc[:, unique]

51 51
106


In [961]:
duplicates = [st for st in rhone_stations_reg1.index if st.endswith('.1')]
print(len([st for st in duplicates if st[:-2] in rhone_stations_reg1.index]), len(duplicates), 'duplicates')
unique = [st for st in rhone_stations_reg1.index if not st.endswith('.1') and not st.startswith('SLF')]
print(len(unique), len(rhone_stations_reg1.index))

df_reg1_updated = pd.DataFrame(index=df_reg1.index)
for st1 in duplicates:
    st0 = st1[:-2]
    data = df_reg1[st0]
    data[data.isnull()] = df_reg1[st1]
    df_reg1_updated.loc[:, st0] = data
df_reg1_updated.loc[:, unique] = df_reg1.loc[:, unique]

27 27 duplicates
60 94


In [ ]:
# region 2
df_prec_reg2 = df_prec.copy()
existing_stations = [col for col in df_prec_reg2.columns if col in df_reg2_updated.columns]
df_prec_reg2[existing_stations] = df_prec_reg2[existing_stations].fillna(df_reg2_updated[existing_stations])

existing_stations = [col for col in df_reg2_updated.columns if col not in df_prec_reg2.columns]
df_prec_reg2.loc[:, existing_stations] = df_reg2_updated.loc[:, existing_stations]


In [ ]:
# region 1
df_prec_reg1 = df_prec.copy()
existing_stations = [col for col in df_prec_reg1.columns if col in df_reg1_updated.columns]
df_prec_reg1[existing_stations] = df_prec_reg1[existing_stations].fillna(df_reg1_updated[existing_stations])

existing_stations = [col for col in df_reg1_updated.columns if col not in df_prec_reg1.columns]
df_prec_reg1.loc[:, existing_stations] = df_reg1_updated.loc[:, existing_stations]


In [967]:
coords = ['station_height_masl','station_coordinates_lv95_east','station_coordinates_lv95_north']
stations_meteoch_rhone.loc[:, coords]
gdf_rhone_stations = gpd.GeoDataFrame(index=stations_meteoch_rhone.index, 
                                      geometry=gpd.points_from_xy(stations_meteoch_rhone['station_coordinates_lv95_east'], 
                                                                  stations_meteoch_rhone['station_coordinates_lv95_north']), crs="EPSG:2056").to_crs("EPSG:21781")
gdf_rhone_stations['x'] = gdf_rhone_stations.geometry.x
gdf_rhone_stations['y'] = gdf_rhone_stations.geometry.y
gdf_rhone_stations['z'] = stations_meteoch_rhone['station_height_masl']

In [968]:
reg2_stations = rhone_stations_reg2.copy()
reg2_stations = gdf_rhone_stations[['z', 'x', 'y']].combine_first(reg2_stations)

In [969]:
reg1_stations = rhone_stations_reg1.copy()
reg1_stations = gdf_rhone_stations[['z', 'x', 'y']].combine_first(reg1_stations)

In [970]:
# write data: region 2

In [972]:
df_export = df_prec_reg2
time = df_export.index
time_cols = pd.DataFrame()
time_cols['YY'] = time.year
time_cols['MM'] = time.month
time_cols['DD'] = time.day
time_cols['HH'] = time.hour+1

df_export = df_export.reset_index().drop(columns=['index'])
df_export = pd.concat([time_cols, df_export], axis=1)
df_export = df_export.fillna(-9999)
df_export = df_export.map(lambda x: '%g' % x)

In [973]:
df_export_coords = reg2_stations.astype(int).loc[df_prec_reg2.columns].T
df_export_coords = df_export_coords.reset_index().drop(columns=['index'])
df_tmp = pd.DataFrame([df_export_coords.columns], columns=df_export_coords.columns)
df_export_coords = pd.concat([df_export_coords, df_tmp])
df_tmp = pd.DataFrame(index=df_export_coords.index)
n = 4
df_tmp['YY'] = ['YY']*n
df_tmp['MM'] = ['MM']*n
df_tmp['DD'] = ['DD']*n
df_tmp['HH'] = ['HH']*n
df_export_coords = pd.concat([df_tmp, df_export_coords], axis=1)

In [974]:
df_export_all = pd.concat([df_export_coords, df_export])

In [976]:
ofile = '/home/aelkouk/hyclimm/Rhone/Input/timeseries/precipitation_reg2_2025.dat'
header = 'Valais_Meteoswiss_station_data\n'
with open(ofile, 'w') as f:
    f.writelines(header)
    df_export_all.to_csv(f, sep="\t", index=False, header=False)

In [977]:
# write data: region 1

In [978]:
df_export = df_prec_reg1
time = df_export.index
time_cols = pd.DataFrame()
time_cols['YY'] = time.year
time_cols['MM'] = time.month
time_cols['DD'] = time.day
time_cols['HH'] = time.hour+1

df_export = df_export.reset_index().drop(columns=['index'])
df_export = pd.concat([time_cols, df_export], axis=1)
df_export = df_export.fillna(-9999)
df_export = df_export.map(lambda x: '%g' % x)

In [979]:
df_export_coords = reg1_stations.astype(int).loc[df_prec_reg1.columns].T
df_export_coords = df_export_coords.reset_index().drop(columns=['index'])
df_tmp = pd.DataFrame([df_export_coords.columns], columns=df_export_coords.columns)
df_export_coords = pd.concat([df_export_coords, df_tmp])
df_tmp = pd.DataFrame(index=df_export_coords.index)
n = 4
df_tmp['YY'] = ['YY']*n
df_tmp['MM'] = ['MM']*n
df_tmp['DD'] = ['DD']*n
df_tmp['HH'] = ['HH']*n
df_export_coords = pd.concat([df_tmp, df_export_coords], axis=1)

In [980]:
df_export_all = pd.concat([df_export_coords, df_export])

In [981]:
ofile = '/home/aelkouk/hyclimm/Rhone/Input/timeseries/precipitation_reg1_2025.dat'
header = 'Valais_Meteoswiss_station_data\n'
with open(ofile, 'w') as f:
    f.writelines(header)
    df_export_all.to_csv(f, sep="\t", index=False, header=False)